# Open a dynamical.org Dataset via STAC + xpystac and Visualize with hvPlot

This notebook builds on the catalog created by **`build_dynamical_catalog_full.ipynb`**.
It shows the full workflow:

1. **Load** the remote STAC catalog with `pystac`
2. **Retrieve** an item and find its icechunk asset
3. **Open** the dataset directly in `xarray` via the `xpystac` backend
4. **Select** a single time step and variable
5. **Visualize** as an interactive map with `hvplot`

```
pip install xpystac icechunk hvplot
```

## 1. Load the STAC catalog and open a dataset

In [ ]:
import pystac, xarray as xr
import xpystac  # registers xarray backend for icechunk stores
import zarr

In [ ]:
zarr.config.set({"async.concurrency": 4})

In [ ]:
catalog = pystac.Catalog.from_file("https://r2-pub.openscicomp.io/stac/dynamical/catalog.json")
item = catalog.get_item("noaa-gfs-analysis-v0-1-0")

asset_key = next(k for k in item.assets if '@' in k)
ds = xr.open_dataset(item.assets[asset_key], engine='stac', chunks=None)
ds

In [ ]:
da = ds['temperature_2m'].sel(time='2026-03-24', longitude=slice(-120,-60), latitude=slice(50,20)).compute()

In [ ]:
da

In [ ]:
import hvplot.xarray

In [ ]:
da.hvplot.image(x='longitude', y='latitude', rasterize=True)

In [ ]:
VAR = "temperature_2m"  # already in °C in this dataset

t_latest = ds.time[-1].values
print(f"Latest available time: {t_latest}")

da = ds[VAR].sel(time=t_latest).load()
print(f"Shape:  {da.shape}")
print(f"Min:    {float(da.min()):.1f} {da.attrs.get('units')}")
print(f"Max:    {float(da.max()):.1f} {da.attrs.get('units')}")
da

## 6. Subset or coarsen before plotting

Plotting the full 1440×721 global grid overwhelms the renderer even with Datashader.
Two options — pick one:

**Option A — spatial subset** (best for regional analysis):
```python
da_plot = da.sel(latitude=slice(lat_max, lat_min), longitude=slice(lon_min, lon_max))
```

**Option B — coarsen** (keeps global view, reduces resolution):
```python
da_plot = da.coarsen(latitude=4, longitude=4, boundary="trim").mean()
```

In [ ]:
# Option A: spatial subset (CONUS)
lat_min, lat_max = 25.0, 55.0
lon_min, lon_max = -130.0, -60.0

da_plot = da.sel(
    latitude=slice(lat_max, lat_min),   # stored descending
    longitude=slice(lon_min, lon_max),
)

# Option B: global view at reduced resolution — uncomment to use instead
# da_plot = da.coarsen(latitude=4, longitude=4, boundary="trim").mean()

print(f"Plot shape: {da_plot.shape}")

## 7. Visualize with hvPlot

In [ ]:
import hvplot.xarray  # noqa — registers .hvplot accessor
import cartopy.crs as crs

In [ ]:
import hvplot.xarray  # noqa

title = (
    f"{da_plot.attrs.get('long_name', VAR)} [{da_plot.attrs.get('units', '')}]\n"
    f"{str(t_latest)[:16]} UTC  |  NOAA GFS Analysis 0.25°"
)

da_plot.hvplot.image(
    x="longitude", y="latitude",
    cmap="RdBu_r",
    clabel=da_plot.attrs.get("units", ""),
    title=title,
    width=700,
    height=400,
    rasterize=True,
)

## 8. Multi-step widget: explore recent time steps

In [ ]:
# Load last 5 time steps for the same spatial extent
da_recent = ds[VAR].isel(time=slice(-5, None)).sel(
    latitude=slice(lat_max, lat_min),
    longitude=slice(lon_min, lon_max),
).load()

da_recent.hvplot.image(
    x="longitude", y="latitude",
    groupby="time",
    cmap="RdBu_r",
    clabel=da_recent.attrs.get("units", ""),
    title=f"NOAA GFS Analysis — {VAR}",
    width=700,
    height=400,
    rasterize=True,
)